# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


**Version complétée avec assistance IA.** Solutions et explications adaptées des [corrigés de Christophe Guyeux](https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/), sous licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). Les blocs du cours nécessaires aux exercices sont ajoutés. Exécuter les cellules dans l’ordre. Les temps mesurés dépendent de la machine. Relire les réponses et indiquer dans le README les vérifications réalisées personnellement.

**Rappel du cours / préparation pour la suite**

In [1]:
print(ord("A"), ord("Z"), chr(72))

65 90 H


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [2]:
def lettre_vers_nombre(c):

    return ord(c) - ord("A")

def nombre_vers_lettre(n):

    return chr(n % 26 + ord("A"))

print(lettre_vers_nombre("C"), nombre_vers_lettre(2))
print([nombre_vers_lettre(lettre_vers_nombre(c)) == c for c in "ABCXYZ"])

2 C
[True, True, True, True, True, True]


Le % 26 dans la seconde fonction n'est pas superflu : il garantit qu'un nombre
hors de l'intervalle attendu retombe sur une lettre valide, ce qui simplifie tout
le reste du TP.

## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [3]:
import unicodedata

def nettoyer(texte):

    sans_accent = unicodedata.normalize("NFD", texte)
    sans_accent = "".join(c for c in sans_accent if unicodedata.category(c) != "Mn")
    return "".join(c for c in sans_accent.upper() if "A" <= c <= "Z")

print(nettoyer("Été, à Paris !"))

ETEAPARIS


La normalisation NFD sépare une lettre accentuée en deux caractères, la lettre
et l'accent ; on jette ensuite les accents, repérés par leur catégorie Unicode
Mn. C'est la manière correcte de faire, bien plus robuste qu'une table de
remplacement écrite à la main, qui oublie toujours un caractère.

## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [4]:
def chiffrer(texte, a, b):

    resultat = []
    for c in nettoyer(texte):
        x = lettre_vers_nombre(c)
        resultat.append(nombre_vers_lettre((a * x + b) % 26))
    return "".join(resultat)

message = "ATTAQUEALAUBE"
print(chiffrer(message, 5, 8))
print(chiffrer(message, 1, 3))   

IZZIKECILIENC
DWWDTXHDODXEH


Avec \( a = 1 \) et \( b = 3 \), chaque lettre est simplement décalée de trois
rangs : c'est le chiffre que Suétone attribue à Jules César.

## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [5]:
def pgcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

valides = [a for a in range(1, 26) if pgcd(a, 26) == 1]
print(valides)
print(len(valides), "valeurs possibles pour a")

[1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
12 valeurs possibles pour a


On obtient [1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]. Comme \( 26 = 2
\times 13 \), il faut exclure tous les multiples de 2 et de 13. Le nombre total de
clés est donc \( 12 \times 26 = 312 \), ce qui est ridiculement petit : retenez ce
chiffre pour l'étape 5.

## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [6]:
def inverse_modulaire(a, n):

    a = a % n
    for candidat in range(1, n):
        if (a * candidat) % n == 1:
            return candidat
    raise ValueError(f"{a} n'est pas inversible modulo {n}")

print(inverse_modulaire(5, 26))
print((5 * 21) % 26)

try:
    inverse_modulaire(4, 26)
except ValueError as erreur:
    print("erreur attendue :", erreur)

21
1
erreur attendue : 4 n'est pas inversible modulo 26


La recherche exhaustive convient parfaitement ici, puisque \( n = 26 \). Pour un
module de trois cents chiffres, comme en cryptographie réelle, elle serait
impraticable et l'on utilise l'algorithme d'Euclide étendu, qui donne l'inverse en
quelques dizaines d'opérations. C'est le même écart qu'entre le test de primalité
naïf et le crible du TP2.

## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [7]:
def dechiffrer(texte, a, b):

    a_inverse = inverse_modulaire(a, 26)
    resultat = []
    for c in nettoyer(texte):
        y = lettre_vers_nombre(c)
        resultat.append(nombre_vers_lettre(a_inverse * (y - b) % 26))
    return "".join(resultat)

message = nettoyer("Attaque a l aube")
secret = chiffrer(message, 5, 8)
print(secret)
print(dechiffrer(secret, 5, 8))
print(dechiffrer(secret, 5, 8) == message)


try:
    dechiffrer(secret, 4, 7)
except ValueError as erreur:
    print("cle refusee :", erreur)

IZZIKECILIENC
ATTAQUEALAUBE
True
cle refusee : 4 n'est pas inversible modulo 26


Le test aller-retour, chiffrer puis déchiffrer et comparer à l'original, est la
bonne façon de vérifier ce genre de code. Il attrape toutes les erreurs de signe
et de modulo d'un seul coup.

## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [8]:
def toutes_les_clefs(secret):

    resultats = []
    for a in valides:
        for b in range(26):
            resultats.append((a, b, dechiffrer(secret, a, b)))
    return resultats

essais = toutes_les_clefs(secret)
print(len(essais), "clefs essayees")
bons = [e for e in essais if e[2] == message]
print("la bonne clef :", bons[0][0], bons[0][1])

312 clefs essayees
la bonne clef : 5 8


Trois cent douze essais, c'est instantané. Comparez avec une clé moderne de 128
bits, qui offre \( 2^{128} \) possibilités, soit un nombre à trente-neuf chiffres :
aucune machine existante ou concevable n'en viendrait à bout. La sécurité d'un
chiffrement tient d'abord à la taille de son espace de clés.

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [9]:
from collections import Counter

texte_long = nettoyer(
    "Le chiffrement affine est un chiffrement par substitution monoalphabetique. "
    "Chaque lettre du message est remplacee par une autre lettre, toujours la meme, "
    "ce qui laisse intactes les frequences relatives des lettres et permet de casser "
    "le code sans essayer toutes les clefs possibles."
)
secret_long = chiffrer(texte_long, 7, 12)

comptes = Counter(secret_long)
lettre_frequente = comptes.most_common(1)[0][0]
print("lettre la plus frequente du chiffre :", lettre_frequente)


cible = lettre_vers_nombre(lettre_frequente)
candidats = []
for a in valides:
    
    b = (cible - 4 * a) % 26
    candidats.append((a, b, dechiffrer(secret_long, a, b)))

print(len(candidats), "candidats au lieu de 312")

frequences = [7.6, 0.9, 3.3, 3.7, 14.7, 1.1, 0.9, 0.7, 7.5, 0.6, 0.1, 5.5, 3.0,
              7.1, 5.8, 2.5, 1.4, 6.6, 7.9, 7.2, 6.3, 1.8, 0.1, 0.4, 0.3, 0.1]
def score_francais(texte):
    return sum(frequences[lettre_vers_nombre(c)] for c in texte) / len(texte)
a, b, texte = max(candidats, key=lambda candidat: score_francais(candidat[2]))
print("cle estimee :", a, b)
print(texte)

assert texte == texte_long


lettre la plus frequente du chiffre : O
12 candidats au lieu de 312
cle estimee : 7 12
LECHIFFREMENTAFFINEESTUNCHIFFREMENTPARSUBSTITUTIONMONOALPHABETIQUECHAQUELETTREDUMESSAGEESTREMPLACEEPARUNEAUTRELETTRETOUJOURSLAMEMECEQUILAISSEINTACTESLESFREQUENCESRELATIVESDESLETTRESETPERMETDECASSERLECODESANSESSAYERTOUTESLESCLEFSPOSSIBLES


On passe de 312 essais à 12, et le bon apparaît immédiatement. La méthode a
cependant besoin d'un texte assez long : sur "ATTAQUEALAUBE", la lettre la plus
fréquente est A et non E, et l'hypothèse tombe à côté. Essayez, c'est instructif.

On peut alors rendre la méthode robuste en testant les trois lettres les plus
fréquentes du chiffré contre les trois du français, ce qui donne neuf hypothèses,
toujours bien moins que 312.

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [10]:
import random

def indice_coincidence(texte):

    N = len(texte)
    comptes = Counter(texte)
    return sum(n * (n - 1) for n in comptes.values()) / (N * (N - 1))

random.seed(1)
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
hasard = "".join(random.choice(alphabet) for _ in range(len(texte_long)))

print(f"clair    : {indice_coincidence(texte_long):.4f}")
print(f"affine   : {indice_coincidence(secret_long):.4f}")
print(f"hasard   : {indice_coincidence(hasard):.4f}")

clair    : 0.0872
affine   : 0.0872
hasard   : 0.0387


Le chiffrement affine ne change pas l'indice : il permute les lettres, donc il
permute les \( n_\ell \) sans changer leur somme. C'est une signature, et elle
suffit à dire, sans même essayer de déchiffrer, qu'un texte a été chiffré par
substitution simple. La valeur est un peu haute ici parce que le texte est court
et répétitif ; sur un roman, elle tombe vers 0,075.

## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [11]:
def vigenere(texte, cle):

    resultat = []
    for i, c in enumerate(nettoyer(texte)):
        decalage = lettre_vers_nombre(cle[i % len(cle)])
        resultat.append(nombre_vers_lettre(lettre_vers_nombre(c) + decalage))
    return "".join(resultat)

secret_vigenere = vigenere(texte_long, "CLE")
print(secret_vigenere[:40])
print(f"IC vigenere : {indice_coincidence(secret_vigenere):.4f}")

NPGJTJHCIOPRVLJHTRGPWVFRESMHQVGXIPETCCWW
IC vigenere : 0.0549


La fonction nombre_vers_lettre de l'étape 1 fait le % 26 pour nous : c'est
pour cela qu'on l'avait écrite ainsi. Avec trois décalages différents mélangés,
l'indice descend, et sur une clé longue il rejoindrait celui du hasard.

## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [12]:
for L in range(1, 8):
    colonnes = [secret_vigenere[i::L] for i in range(L)]
    moyenne = sum(indice_coincidence(col) for col in colonnes) / L
    print(f"L = {L} : indice moyen {moyenne:.4f}")

L = 1 : indice moyen 0.0549
L = 2 : indice moyen 0.0557
L = 3 : indice moyen 0.0916
L = 4 : indice moyen 0.0539
L = 5 : indice moyen 0.0536
L = 6 : indice moyen 0.0918
L = 7 : indice moyen 0.0558


La notation texte[i::L] prend une lettre sur L à partir de la position i.
Une fois la longueur connue, chaque colonne se casse comme un César, par la
méthode de l'étape 6 : la lettre la plus fréquente de la colonne est l'image de E,
ce qui donne le décalage, donc la lettre de la clé. Vous avez tous les outils pour
finir : faites-le, et vous aurez cassé sans aucune indication un chiffre resté
réputé incassable pendant trois siècles.

Ce qui l'a finalement mis à terre, c'est exactement ce que vous venez de faire :
une statistique, l'indice de coïncidence, et un peu de calcul. C'est aussi la
raison pour laquelle les chiffrements modernes sont conçus pour qu'aucune
régularité du clair ne survive dans le chiffré.

**Note de lecture :** les commentaires du corrigé donnent des ordres de grandeur. Pour les durées et les erreurs numériques, les sorties enregistrées dans ce carnet sont les mesures de cette exécution ; elles peuvent varier en relançant le carnet.